# 03 — Numerical ODE Solving with JAX

An ordinary differential equation (ODE): dy/dt = f(y, t)

JAX's `lax.scan` makes ODE solvers:
1. **JIT-compilable** — no Python overhead at solve time
2. **Differentiable** — gradients flow through the solver (useful for physics-informed NNs, parameter fitting)
3. **Batchable** — `vmap` over initial conditions at no extra coding cost

**Connection to RNNs**: A discretised ODE _is_ an RNN step — h_{t+1} = h_t + dt·f(h_t, u_t).

In [ ]:
import sys; sys.path.insert(0, '..')
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

from src.ode_solvers import solve_euler, solve_rk4, solve_rk4_batch
from src.ode_solvers import harmonic_oscillator, lorenz, van_der_pol

jax.config.update('jax_enable_x64', True)

## 3.1 Euler vs RK4: Accuracy Comparison

Solve the harmonic oscillator x'' + ω²x = 0 (state: [x, v]).

In [ ]:
omega = 1.0
y0 = jnp.array([1.0, 0.0])   # x=1, v=0
t = jnp.linspace(0, 10, 500)

f = lambda y, t: harmonic_oscillator(y, t, omega=omega)

# Solve with both methods
ys_euler = solve_euler(f, y0, t)
ys_rk4   = solve_rk4(f, y0, t)

# Reference: analytical solution x(t) = cos(ω t)
x_exact = jnp.cos(omega * t[1:])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t[1:], x_exact, 'k-', lw=2, label='exact')
axes[0].plot(t[1:], ys_euler[:, 0], '--', label='Euler')
axes[0].plot(t[1:], ys_rk4[:, 0], ':', lw=2, label='RK4')
axes[0].set(xlabel='t', ylabel='x(t)', title='Position')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(t[1:], jnp.abs(ys_euler[:, 0] - x_exact), label='Euler error')
axes[1].semilogy(t[1:], jnp.abs(ys_rk4[:, 0] - x_exact), label='RK4 error')
axes[1].set(xlabel='t', ylabel='|error|', title='Absolute error (log scale)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

euler_err = float(jnp.abs(ys_euler[-1, 0] - x_exact[-1]))
rk4_err   = float(jnp.abs(ys_rk4[-1, 0] - x_exact[-1]))
print(f'Final error — Euler: {euler_err:.4e}  RK4: {rk4_err:.4e}')

## 3.2 Solving the Lorenz System

In [ ]:
from functools import partial

y0_lorenz = jnp.array([1.0, 0.0, 0.0])
t_lorenz = jnp.linspace(0, 30, 10000)

# Use functools.partial to close over the ODE function, then JIT over arrays only.
# (jax.jit requires non-array args to be static; partial pre-binds them.)
ys_lorenz = jax.jit(partial(solve_rk4, lorenz))(y0_lorenz, t_lorenz)

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(121, projection='3d')
ax.plot(*ys_lorenz.T, lw=0.4, alpha=0.8)
ax.set_title('Lorenz attractor')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

ax2 = fig.add_subplot(122)
ax2.plot(t_lorenz[1:], ys_lorenz[:, 0], lw=0.6)
ax2.set(xlabel='t', ylabel='x(t)', title='x vs time')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3.3 `vmap` over Initial Conditions

In scientific computing, you often want to solve the _same_ ODE for thousands of different initial conditions (Monte Carlo, sensitivity analysis, ensemble forecasting). `vmap` makes this trivial.

In [ ]:
import time
from functools import partial

key = jax.random.PRNGKey(0)
n_ic = 200   # 200 initial conditions
y0_batch = jax.random.uniform(key, (n_ic, 2), minval=-1.5, maxval=1.5)  # (N, 2)

t_short = jnp.linspace(0, 5, 300)

# Close over the ODE function with partial, then JIT over the array args.
batch_solve = jax.jit(partial(solve_rk4_batch, harmonic_oscillator))
_ = batch_solve(y0_batch, t_short)  # warm up (triggers JIT compilation)

t0 = time.perf_counter()
trajs = batch_solve(y0_batch, t_short)
_ = trajs.block_until_ready()
elapsed = time.perf_counter() - t0

print(f'Solved {n_ic} trajectories in {elapsed*1000:.1f} ms')
print(f'Output shape: {trajs.shape}  (n_ic, T-1, D)')

In [ ]:
# Phase-space portrait: many trajectories at once
plt.figure(figsize=(7, 6))
for i in range(min(50, n_ic)):
    plt.plot(trajs[i, :, 0], trajs[i, :, 1], alpha=0.4, lw=0.8)
plt.scatter(y0_batch[:50, 0], y0_batch[:50, 1], c='red', s=20, zorder=5, label='ICs')
plt.xlabel('x (position)'); plt.ylabel('v (velocity)')
plt.title('Harmonic oscillator: 50 trajectories (vmap over ICs)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3.4 Differentiating Through the Solver

Because `solve_rk4` is built from `lax.scan` and pure JAX ops, gradients flow through it automatically.

In [ ]:
# Example: fit ω² of a harmonic oscillator from trajectory data
t_fit = jnp.linspace(0, 5, 100)
y0_fit = jnp.array([1.0, 0.0])
true_omega = 2.0

# Noisy data
data = jnp.cos(true_omega * t_fit) + 0.05 * jax.random.normal(jax.random.PRNGKey(0), t_fit.shape)

def model_loss(log_omega):
    omega = jnp.exp(log_omega)
    f_param = lambda y, t: harmonic_oscillator(y, t, omega=omega)
    ys = solve_rk4(f_param, y0_fit, t_fit)
    return jnp.mean((ys[:, 0] - data[1:]) ** 2)

log_omega = jnp.log(jnp.array(1.0))   # start at ω=1
grad_loss = jax.jit(jax.grad(model_loss))

losses = []
for _ in range(100):
    g = grad_loss(log_omega)
    log_omega = log_omega - 0.1 * g
    losses.append(float(model_loss(log_omega)))

print(f'True ω: {true_omega:.3f}   Fitted ω: {float(jnp.exp(log_omega)):.3f}')

plt.figure(figsize=(7, 3))
plt.semilogy(losses); plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Fitting ω via gradient descent through the ODE solver'); plt.grid(alpha=0.3)
plt.show()

## 3.5 Connection to RNNs

An RNN is a discretised ODE with Euler method:

$$h_{t+1} = h_t + \Delta t \cdot \tanh(W_{\text{rec}} h_t + W_{\text{in}} u_t)$$

Setting Δt = 1 gives the standard Elman RNN. This framing motivates **continuous-time RNNs** (CTRNNs) and **neural ODEs**.

In [ ]:
def ctrnn_step(h, u, W_rec, W_in, tau=1.0, dt=0.1):
    """One step of a continuous-time RNN (Euler discretisation)."""
    dh = (-h + jnp.tanh(W_rec @ h + W_in @ u)) / tau
    return h + dt * dh

D, I, T = 16, 2, 200
key = jax.random.PRNGKey(0)
W_rec = jax.random.normal(key, (D, D)) * 0.3
W_in = jax.random.normal(jax.random.PRNGKey(1), (D, I))
inputs = jax.random.normal(jax.random.PRNGKey(2), (T, I))

def run_ctrnn(inputs):
    def step(h, u):
        h_new = ctrnn_step(h, u, W_rec, W_in)
        return h_new, h_new
    _, hs = jax.lax.scan(step, jnp.zeros(D), inputs)
    return hs

hs = jax.jit(run_ctrnn)(inputs)
print('CTRNN hidden states shape:', hs.shape)   # (T, D)

plt.figure(figsize=(10, 3))
plt.plot(hs[:, :4])
plt.xlabel('t'); plt.ylabel('h'); plt.title('CTRNN hidden state trajectories (4 units)')
plt.grid(alpha=0.3); plt.show()